# Get start with the Colette Python API

## Install the ipywidgets package if not already installed

In [ ]:
# !pip install -U ipywidgets

## Import necessary libraries

In [ ]:
import json
import base64
from io import BytesIO
from PIL import Image
from IPython.display import display
import re

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from colette.jsonapi import JSONApi
from colette.apidata import APIData

## Get paths and initialize variables

In [ ]:
# Get the root path of the colette package
import os
colette_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
# print(f'Colette root path: {colette_root}')

In [ ]:
documents_dir = os.path.join(colette_root, 'docs/pdf') # where the input documents are located
app_dir = os.path.join(colette_root, 'app_colette') # where to store the app
models_dir = os.path.join(colette_root, 'models') # where the models are located

config_file = os.path.join(colette_root, 'src/colette/config/vrag_default.json')
index_file = os.path.join(colette_root, 'src/colette/config/vrag_default_index.json')

app_name = 'app_colette' # name of the app

In [ ]:
colette_api = JSONApi()

## Get config files and set parameters

In [ ]:
# read the configuration file
with open(config_file, 'r') as f:
    create_config = json.load(f)
with open(index_file, 'r') as f:
    index_config = json.load(f)

create_config['app']['repository'] = app_dir
create_config['app']['models_repository'] = models_dir
index_config['parameters']['input']['data'] = [documents_dir]
#index_config['parameters']['input']['rag']['reindex'] = False # if True, the RAG will be reindexed


## Create the Service API client

In [ ]:
api_data_create = APIData(**create_config)
colette_api.service_create(app_name, api_data_create)

## Index the documents

In [ ]:
# index the documents
api_data_index = APIData(**index_config)
colette_api.service_index(app_name, api_data_index)

## Query the documents

Note the optional 'crop_label' parameter to filter the sources by crop label.

The default crop labels are: 'text', 'table', 'figure'

In [ ]:
# query the vision RAG
query_api_msg = {
    'parameters': {
        'input': {
            'message': 'What are the identified sources of errors ?',
            # 'crop_label': 'text',  # optional, to specify a crop label
        }
    }
}

query_data = APIData(**query_api_msg)
response = colette_api.service_predict(app_name, query_data)

In [ ]:
print(response.output)

In [ ]:
# Function to extract and display image from base64 data URI
def display_image_from_data_uri(data_uri):
    # Extract base64 string (remove 'data:image/png;base64,' prefix)
    base64_str = re.sub('^data:image/.+;base64,', '', data_uri)
    
    # Decode base64 string
    image_data = base64.b64decode(base64_str)
    
    # Create PIL Image
    image = Image.open(BytesIO(image_data))
    
    # Display image
    display(image)
    
    return image

In [ ]:
# Display all images in your context
for item in response.sources['context']:
    print(f"Key: {item['key']}")
    print(f"Distance: {item['distance']}")
    print(f'crop_label: {item.get("crop_label", "N/A")}')
    image = display_image_from_data_uri(item['content'])
    print(f"Image size: {image.size}")
    print("-" * 50)